In [67]:
import pandas as pd

In [68]:
df = pd.read_json(
    'C:/repository/TCC/data/raw/dados_artigos.json')

In [69]:
df.head(1)

,titulo,data,doi,volume,autores,keywords,url
0,Analysis of Expenses from Brazilian Federal De...,Data Desconhecida,10.5753/jidm.2025.3383,Vol. 16 No. 1 (2025),"[{'nome': 'Felippe Pires Ferreira', 'inst': 'U...","[Analysis of expenses, Public expenses, Brazil...",https://journals-sol.sbc.org.br/index.php/jidm...


In [70]:
df = df.explode('autores')

autores = pd.json_normalize(df['autores'])

df = df.drop(columns=['autores', 'url', 'doi']).reset_index(drop=True)

In [71]:
df['instituicao'] = autores.groupby(level=0).agg({'inst': lambda x: ', '.join(x)})

In [72]:
df['autores'] = autores.groupby(level=0).agg({'nome': lambda x: ', '.join(x)})

In [73]:
keywords = df.explode('keywords')


In [74]:
df["keywords"] = (
    keywords.groupby(level=0)
            .agg({'keywords': lambda x: ', '.join(x.dropna().astype(str))})
)

In [ ]:
import re
import unicodedata


def pre_processar(texto):
    if not texto:
        return ""
    if isinstance(texto, float):
        return ""

    texto = texto.upper()
    texto = ''.join(c for c in unicodedata.normalize('NFKD', texto)
                    if unicodedata.category(c) != 'Mn')
    return texto


def normalizar_instituicao(nome_bruto):
    nome = pre_processar(nome_bruto)

    mapeamento_prioritario = [
        (r".*(FLUMIN|UFF|INFES/UFF|IC/UFF|PPGCI/UFF).*",
         "Universidade Federal Fluminense (UFF)"),
        (r".*(SAO PAULO|USP|ICMC|POLITECNICA|EACH).*",
         "Universidade de São Paulo (USP)"),
        (r".*(MINAS GERAIS|UFMG|BIOINFORMATICA|DCC/UFMG).*",
         "Universidade Federal de Minas Gerais (UFMG)"),
        (r".*(RIO DE JANEIRO|UFRJ|COPPE|PPGI/UFRJ|BNDES|IBCCF|PPGEN|CISI/COPPE).*",
         "Universidade Federal do Rio de Janeiro (UFRJ)"),
        (r".*(CAMPINA GRANDE|UFCG|CAMINA GRANDE).*",
         "Universidade Federal de Campina Grande (UFCG)"),
        (r".*(CAMPINAS|UNICAMP|CEPAGRI).*",
         "Universidade Estadual de Campinas (UNICAMP)"),
        (r".*(CEARA|UFC).*",
         "Universidade Federal do Ceará (UFC)"),
        (r".*(PERNAMBUCO|UFPE|CIN/UFPE|UFRPE|AGRESTE).*",
         "Universidade Federal de Pernambuco (UFPE)"),
        (r".*(SAO CARLOS|UFSCAR).*",
         "Universidade Federal de São Carlos (UFSCar)"),
        (r".*(PUC RIO|PONTIFICAL CATHOLIC UNIVERSITY OF RIO DE JANEIRO).*",
         "PUC-Rio"),
        (r".*(GOIAS|UFG).*",
         "Universidade Federal de Goiás (UFG)"),
        (r".*(PARANA|UFPR).*",
         "Universidade Federal do Paraná (UFPR)"),
        (r".*(UBERLANDIA|UFU).*",
         "Universidade Federal de Uberlândia (UFU)"),
        (r".*(OURO PRETO|UFOP).*",
         "Universidade Federal de Ouro Preto (UFOP)"),
        (r".*(SANTA CATARINA|UFSC).*",
         "Universidade Federal de Santa Catarina (UFSC)"),
        (r".*(RIO GRANDE DO SUL|UFRGS).*",
         "Universidade Federal do Rio Grande do Sul (UFRGS)"),
        (r".*(AMAZONAS|UFAM).*",
         "Universidade Federal do Amazonas (UFAM)"),
        (r".*(BRASILIA|UNB).*",
         "Universidade de Brasília (UnB)"),
        (r".*(SAO JOAO DEL REI|UFSJ).*",
         "Universidade Federal de São João Del Rei (UFSJ)"),
        (r".*(FORTALEZA|UNIFOR).*",
         "Universidade de Fortaleza (UNIFOR)"),
        (r".*(INSTITUTO FEDERAL|IFSP|IFMG|IFPR|IFNMG|IFPB|IFRS|IFSUL|IF SUDOESTE).*",
         "Institutos Federais (IFs)"),
        (r".*(CEFET).*",
         "CEFET"),
        (r".*(UTFPR|TECHNOLOGY PARANA).*",
         "UTFPR"),
        (r".*(AERONAUTICA|ITA).*",
         "Instituto Tecnológico de Aeronáutica (ITA)"),
        (r".*(FIOCRUZ|OSWALDO CRUZ).*",
         "Fundação Oswaldo Cruz (Fiocruz)"),
        (r".*(EMBRAPA).*",
         "Embrapa"),
        (r".*(IBM).*",
         "IBM Research"),
        (r".*(JUSBRASIL).*",
         "Jusbrasil"),
    ]

    for padrao, nome_padrao in mapeamento_prioritario:
        novo_nome = re.sub(padrao, nome_padrao, nome)
        if novo_nome != nome:
            return novo_nome

    return nome_bruto

In [76]:
def pre_processar(texto):
    if pd.isna(texto):
        return ""

    texto = str(texto).upper()

    texto = ''.join(
        c for c in unicodedata.normalize('NFKD', texto)
        if unicodedata.category(c) != 'Mn'
    )

    texto = re.sub(r'[^A-Z0-9\s/]', ' ', texto)

    return ' '.join(texto.split())

In [77]:
df['inst_clean'] = df['instituicao'].apply(pre_processar)

In [78]:
df.head(2)

,titulo,data,volume,keywords,instituicao,autores,inst_clean
0,Analysis of Expenses from Brazilian Federal De...,Data Desconhecida,Vol. 16 No. 1 (2025),"Analysis of expenses, Public expenses, Brazili...","Universidade de São Paulo, Universidade de São...","Felippe Pires Ferreira, Ilan S. G. de Figueire...",UNIVERSIDADE DE SAO PAULO UNIVERSIDADE DE SAO ...
1,Analysis of Expenses from Brazilian Federal De...,Data Desconhecida,Vol. 16 No. 1 (2025),"Analysis of expenses, Public expenses, Brazili...","University of São Paulo, Carnegie Mellon Unive...","João Augusto Fernandes Barbosa, Robson Leonard...",UNIVERSITY OF SAO PAULO CARNEGIE MELLON UNIVER...


In [79]:
df['instituicao'] = df['inst_clean'].apply(normalizar_instituicao)

In [80]:
df = df.drop(columns=['inst_clean'])

In [81]:
df.head(15)

,titulo,data,volume,keywords,instituicao,autores
0,Analysis of Expenses from Brazilian Federal De...,Data Desconhecida,Vol. 16 No. 1 (2025),"Analysis of expenses, Public expenses, Brazili...",Universidade de São Paulo (USP),"Felippe Pires Ferreira, Ilan S. G. de Figueire..."
1,Analysis of Expenses from Brazilian Federal De...,Data Desconhecida,Vol. 16 No. 1 (2025),"Analysis of expenses, Public expenses, Brazili...",Universidade de São Paulo (USP),"João Augusto Fernandes Barbosa, Robson Leonard..."
2,Analysis of Expenses from Brazilian Federal De...,Data Desconhecida,Vol. 16 No. 1 (2025),"Analysis of expenses, Public expenses, Brazili...",Universidade de São Paulo (USP),"Braulio V. Sánchez Vinces, Robson L. F. Cordeiro"
3,Analysis of Expenses from Brazilian Federal De...,Data Desconhecida,Vol. 16 No. 1 (2025),"Analysis of expenses, Public expenses, Brazili...",UNIVERSITY OF FEIRA DE SANTANA UNIVERSITY OF F...,"José Solenir Lima Figuerêdo, Ana Lúcia Marreir..."
4,Analysis of Expenses from Brazilian Federal De...,Data Desconhecida,Vol. 16 No. 1 (2025),"Analysis of expenses, Public expenses, Brazili...",Universidade de São Paulo (USP),"Renato Okabayashi Miyaji, Felipe Valencia de A..."
5,Analysis of Expenses from Brazilian Federal De...,Data Desconhecida,Vol. 16 No. 1 (2025),"Analysis of expenses, Public expenses, Brazili...",Universidade Federal do Ceará (UFC),"Ronildo Oliveira da Silva, Regis Pires Magalhã..."
6,Analysis of Expenses from Brazilian Federal De...,Data Desconhecida,Vol. 16 No. 1 (2025),"Analysis of expenses, Public expenses, Brazili...",Universidade Federal de São Paulo (UNIFESP),"João Pedro de Carvalho Castro, Maria Júlia Soa..."
7,Exploratory Analysis of Microdata from the Nat...,Data Desconhecida,Vol. 16 No. 1 (2025),"Enem, Disability, Exploratory data analysis, E...",Universidade Federal de Pernambuco (UFPE),"Mariama C. S. de Oliveira, Lucas Henrique G. d..."
8,Exploratory Analysis of Microdata from the Nat...,Data Desconhecida,Vol. 16 No. 1 (2025),"Enem, Disability, Exploratory data analysis, E...",UTFPR,"Nicolas Tamalu, Leandro Augusto Ensina, Eduard..."
9,Grid-Ordering for Outlier Detection in Massive...,Data Desconhecida,Vol. 16 No. 1 (2025),"Distance-based outlier detection, Distributed ...",Universidade Federal de Minas Gerais (UFMG),"Claudio M. V. de Andrade, Fabiano Muniz Belém,..."


In [82]:
contagem = df['instituicao'].value_counts()
print(contagem)

instituicao
N/A                                                                               1311
Universidade Federal de Minas Gerais (UFMG)                                         53
Universidade de São Paulo (USP)                                                     49
Universidade Federal do Ceará (UFC)                                                 24
Universidade Federal de São Carlos (UFSCar)                                         19
                                                                                  ... 
CEFET MG UFMG UFMG                                                                   1
NO AFFILIATION DECLARED UFMG UFMG NO AFFILIATION DECLARED                            1
UFMG UFMG UFMG UFMG                                                                  1
SAP GERMANY UNIVERSITY OF KAISERSLAUTERN SAP GERMANY INSTITUICAO NAO INFORMADA       1
UNIVERSITY OF KAISERSLAUTERN SAP GERMANY                                             1
Name: count, Length: 140, dtype